# Goldfish English XNLI — Full Fine-Tuning for Three-Class Classification

This notebook fine-tunes the English Goldfish GPT-2-style checkpoint on the English configuration of `facebook/xnli` as a **three-class sequence-classification task**. It does not use causal language-modelling loss or LoRA: all model parameters, including the newly initialized classification head, are trainable.

**Input format:** `[CLS] premise [SEP] hypothesis [SEP]` (using the Goldfish tokenizer's own BOS/EOS tokens). The final separator is deliberately retained because GPT-2 sequence classification pools the final non-padding token.

Run this notebook independently for each seed (recommended: `42`, `123`, `456`). Each run must start from the original pretrained checkpoint, not from a previously fine-tuned checkpoint.


## 1. Install / import dependencies

Use compatible, recent versions of `transformers`, `datasets`, `accelerate`, `torch`, and `scikit-learn`. If you install or upgrade packages in this notebook, restart the kernel before continuing.


In [ ]:
import os
import json
import random
from pathlib import Path

import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Experiment configuration

Keep the sequence length at 512 for the Goldfish comparison. The batch size below is a starting point; reduce it if GPU memory is insufficient. Gradient accumulation controls the effective batch size.


In [ ]:
MODEL_ID = "goldfish-models/eng_latn_1000mb"
DATASET_ID = "facebook/xnli"
LANGUAGE_CONFIG = "en"

SEED = 42  # Repeat the complete run with 123 and 456.
MAX_LENGTH = 512
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 1
WARMUP_RATIO = 0.06

OUTPUT_DIR = f"./goldfish-en-xnli-full-ft-seed-{SEED}"


def seed_everything(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    set_seed(seed)


seed_everything(SEED)


## 3. Load tokenizer and verify Goldfish special tokens

Do **not** overwrite the Goldfish padding token with EOS. The model uses separate BOS/CLS, EOS/SEP, and PAD tokens. The assertions below stop the run if the checkpoint does not expose the expected token setup; inspect the tokenizer rather than inventing replacement IDs.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("BOS:", tokenizer.bos_token, tokenizer.bos_token_id)
print("EOS:", tokenizer.eos_token, tokenizer.eos_token_id)
print("PAD:", tokenizer.pad_token, tokenizer.pad_token_id)

if tokenizer.bos_token_id is None:
    raise ValueError("Goldfish tokenizer has no BOS token; inspect its special-token configuration.")
if tokenizer.eos_token_id is None:
    raise ValueError("Goldfish tokenizer has no EOS/SEP token; inspect its special-token configuration.")
if tokenizer.pad_token_id is None:
    raise ValueError("Goldfish tokenizer has no dedicated PAD token. Do not silently alias PAD to EOS.")

BOS_ID = tokenizer.bos_token_id
SEP_ID = tokenizer.eos_token_id
PAD_ID = tokenizer.pad_token_id


## 4. Load XNLI and inspect label metadata

Use the dataset's own `ClassLabel` metadata to establish the class order. Do not assume that numeric label IDs mean a particular class without checking.


In [ ]:
xnli = load_dataset(DATASET_ID, LANGUAGE_CONFIG)

print(xnli)
label_feature = xnli["train"].features["label"]
if not hasattr(label_feature, "names") or label_feature.names is None:
    raise TypeError("Expected the XNLI label column to be a ClassLabel with named classes.")

label_names = list(label_feature.names)
num_labels = len(label_names)
if num_labels != 3:
    raise ValueError(f"Expected 3 NLI classes, found {num_labels}: {label_names}")

label2id = {name: i for i, name in enumerate(label_names)}
id2label = {i: name for i, name in enumerate(label_names)}
print("Label order:", id2label)

for split in ("train", "validation", "test"):
    if split not in xnli:
        raise KeyError(f"Expected XNLI split {split!r} is missing.")
    print(split, len(xnli[split]))


## 5. Tokenize premise–hypothesis pairs

The sequence is constructed explicitly as:

`BOS + premise tokens + SEP + hypothesis tokens + SEP`

We tokenize the two texts separately and truncate the longer side first, reserving space for BOS and both separators. This guarantees that the final SEP survives truncation and remains the final non-padding token. Padding is deferred to the batch collator.


In [ ]:
def encode_nli_batch(examples):
    all_input_ids = []
    all_attention_masks = []

    for premise, hypothesis in zip(examples["premise"], examples["hypothesis"]):
        premise_ids = tokenizer.encode(premise, add_special_tokens=False)
        hypothesis_ids = tokenizer.encode(hypothesis, add_special_tokens=False)

        # Reserve 3 positions: BOS, the premise/hypothesis separator, and final SEP.
        token_budget = MAX_LENGTH - 3
        while len(premise_ids) + len(hypothesis_ids) > token_budget:
            if len(premise_ids) > len(hypothesis_ids):
                premise_ids.pop()
            else:
                hypothesis_ids.pop()

        input_ids = [BOS_ID] + premise_ids + [SEP_ID] + hypothesis_ids + [SEP_ID]
        all_input_ids.append(input_ids)
        all_attention_masks.append([1] * len(input_ids))

    return {
        "input_ids": all_input_ids,
        "attention_mask": all_attention_masks,
        "labels": examples["label"],
    }

remove_columns = xnli["train"].column_names
tokenized_dataset = xnli.map(
    encode_nli_batch,
    batched=True,
    remove_columns=remove_columns,
    desc="Tokenizing XNLI premise/hypothesis pairs",
)

# Sanity checks: the final unpadded token must always be SEP.
for i in range(min(5, len(tokenized_dataset["train"]))):
    ids = tokenized_dataset["train"][i]["input_ids"]
    assert ids[0] == BOS_ID
    assert ids[-1] == SEP_ID
    assert len(ids) <= MAX_LENGTH

print("Example token length:", len(tokenized_dataset["train"][0]["input_ids"]))


## 6. Build a full fine-tuning sequence-classification model

`AutoModelForSequenceClassification` adds a three-class classification head. That head is newly initialized; the pretrained Goldfish backbone is loaded from the checkpoint. No PEFT/LoRA wrapper is used, so all parameters are optimized.

For GPT-2 sequence classification, the model pools the final non-padding token. `pad_token_id` is therefore set explicitly, and the preprocessing above ensures that the final non-padding token is SEP.


In [ ]:
config = AutoConfig.from_pretrained(MODEL_ID)
config.num_labels = num_labels
config.id2label = id2label
config.label2id = label2id
config.pad_token_id = PAD_ID

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    config=config,
)

# Disable KV caching during training.
if hasattr(model.config, "use_cache"):
    model.config.use_cache = False

# Confirm full fine-tuning: all model parameters should be trainable.
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)")
assert trainable_params == total_params, "Some parameters are frozen; this is not full fine-tuning."


## 7. Dynamic padding and evaluation metrics

Dynamic padding reduces wasted computation compared with padding every example to 512 tokens in advance. Macro-F1 is used for checkpoint selection so that each NLI class contributes equally.


In [ ]:
use_fp16 = torch.cuda.is_available()
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if use_fp16 else None,
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        labels=list(range(num_labels)),
        zero_division=0,
    )
    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_precision": float(np.mean(precision)),
        "macro_recall": float(np.mean(recall)),
        "macro_f1": float(np.mean(f1)),
        **{f"{label_names[i]}_precision": float(precision[i]) for i in range(num_labels)},
        **{f"{label_names[i]}_recall": float(recall[i]) for i in range(num_labels)},
        **{f"{label_names[i]}_f1": float(f1[i]) for i in range(num_labels)},
    }


## 8. Fine-tune and select the best checkpoint on validation macro-F1

The values below are starting hyperparameters. If you tune them, use validation data only and apply a comparable tuning budget to Apertus. Do not use the test set for model selection.


In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    fp16=use_fp16,
    logging_steps=100,
    save_total_limit=2,
    seed=SEED,
    data_seed=SEED,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation metric:", trainer.state.best_metric)


## 9. Final held-out test evaluation

This is the final evaluation for this seed. The test split is not used for checkpoint selection.


In [ ]:
test_metrics = trainer.evaluate(tokenized_dataset["test"], metric_key_prefix="test")
print(json.dumps(test_metrics, indent=2))

pred_output = trainer.predict(tokenized_dataset["test"])
test_predictions = np.argmax(pred_output.predictions, axis=-1)
test_labels = pred_output.label_ids

print("Classification report:")
print(
    classification_report(
        test_labels,
        test_predictions,
        labels=list(range(num_labels)),
        target_names=label_names,
        digits=4,
        zero_division=0,
    )
)

cm = confusion_matrix(test_labels, test_predictions, labels=list(range(num_labels)))
print("Confusion matrix (rows=true, columns=predicted):")
print(cm)


## 10. Save model, tokenizer, and run metadata

The saved model directory contains the full fine-tuned model (not an adapter-only checkpoint), tokenizer, and model configuration.


In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

metadata = {
    "model_id": MODEL_ID,
    "dataset_id": DATASET_ID,
    "language_config": LANGUAGE_CONFIG,
    "seed": SEED,
    "max_length": MAX_LENGTH,
    "num_epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "label_order": label_names,
    "trainable_parameters": int(trainable_params),
    "total_parameters": int(total_params),
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_validation_metric": trainer.state.best_metric,
    "test_metrics": {k: float(v) for k, v in test_metrics.items()},
}

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
with open(Path(OUTPUT_DIR) / "run_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("Saved full fine-tuned model and metadata to:", OUTPUT_DIR)


## 11. Notes for the Russian and Chinese runs

For the other Goldfish baselines, change only the model ID, XNLI language configuration, and output directory, then rerun from the beginning with the same seed schedule:

- Russian: `goldfish-models/rus_cyrl_1000mb`, XNLI config `ru`
- Chinese: `goldfish-models/zho_hans_1000mb`, XNLI config `zh`

Each language must use its own Goldfish tokenizer and special-token IDs. Recheck BOS/EOS/PAD and label metadata for each configuration. Do not reuse the English tokenizer. Use the same underlying XNLI examples/labels and evaluation protocol wherever the dataset configurations provide aligned translations.

**Reporting note:** this notebook uses full fine-tuning for Goldfish. If Apertus is fine-tuned with LoRA due to its size, report the different adaptation strategies explicitly; they are not an identical fine-tuning regime.
